In [1]:
import pandas as pd 
df = pd.read_csv("/kaggle/input/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance/PdM_failures.csv")
df2 = pd.read_csv("/kaggle/input/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance/PdM_errors.csv")
df3= pd.read_csv("/kaggle/input/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance/PdM_maint.csv")
df4=pd.read_csv("/kaggle/input/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance/PdM_telemetry.csv")
df5=pd.read_csv("/kaggle/input/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance/PdM_machines.csv")

df["failure"].dtype


dtype('O')

In [2]:
df2["errors_till_now"]=df2.groupby("machineID").cumcount()+1
    

In [3]:
df6 = pd.merge(df4, df, on=["datetime","machineID"],how="left")
df7 = pd.merge(df6, df3, on=["datetime","machineID"],how="left")
df8 = pd.merge(df7, df2, on=["datetime","machineID"],how="left")
df9 = pd.merge(df8, df5, on=["machineID"],how="left")
df9["model"] = df9["model"].astype("category")
df9["errorID"] = df9["errorID"].astype("category")
df9["comp"] = df9["comp"].astype("category")
df9["failure"].dtype


dtype('O')

In [4]:
import numpy as np
df9["Total_hours_from_last_maintenance"] = np.nan
df9["errors_till_now"]=(
    df9.groupby("machineID")["errors_till_now"].ffill()
).fillna(0)

In [5]:
df9["datetime"]=pd.to_datetime(df9["datetime"])

df9 = df9.sort_values(["machineID","datetime"]).reset_index(drop=True)

last_maintenance = (
    df9["datetime"]
    .where(df9["comp"].notna())
    .groupby(df9["machineID"])
    .ffill()
)

df9["Total_hours_from_last_maintenance"]=((
    df9["datetime"]-last_maintenance
).dt.total_seconds()/3600).fillna(0)
df9.head(500)


,datetime,machineID,volt,rotate,pressure,vibration,failure,comp,errorID,errors_till_now,model,age,Total_hours_from_last_maintenance
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,NaN,NaN,NaN,0.0,model3,18,0.0
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,NaN,NaN,NaN,0.0,model3,18,0.0
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,NaN,NaN,NaN,0.0,model3,18,0.0
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,NaN,NaN,NaN,0.0,model3,18,0.0
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,NaN,NaN,NaN,0.0,model3,18,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2015-01-21 19:00:00,1,183.908981,514.869649,99.491418,42.163890,NaN,NaN,NaN,4.0,model3,18,37.0
496,2015-01-21 20:00:00,1,170.692019,465.434954,93.524286,36.413510,NaN,NaN,NaN,4.0,model3,18,38.0
497,2015-01-21 21:00:00,1,162.435715,455.223633,96.753138,46.716736,NaN,NaN,NaN,4.0,model3,18,39.0
498,2015-01-21 22:00:00,1,165.493259,335.843537,100.454325,35.527658,NaN,NaN,NaN,4.0,model3,18,40.0


In [6]:
df9["comp"].dtype

CategoricalDtype(categories=['comp1', 'comp2', 'comp3', 'comp4'], ordered=False, categories_dtype=object)

In [7]:
df9["errorID"] = (
    df9["errorID"]
    .cat.add_categories("no_error")
    .fillna("no_error")
)

df9["comp"]=(
    df9["comp"]
    .cat.add_categories("no_maintenance")
    .fillna("no_maintenance")
)
df9["failure"]=df9["failure"].notna().astype(int)

In [8]:
df9.head(500)

,datetime,machineID,volt,rotate,pressure,vibration,failure,comp,errorID,errors_till_now,model,age,Total_hours_from_last_maintenance
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,0,no_maintenance,no_error,0.0,model3,18,0.0
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,0,no_maintenance,no_error,0.0,model3,18,0.0
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,0,no_maintenance,no_error,0.0,model3,18,0.0
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,0,no_maintenance,no_error,0.0,model3,18,0.0
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,0,no_maintenance,no_error,0.0,model3,18,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2015-01-21 19:00:00,1,183.908981,514.869649,99.491418,42.163890,0,no_maintenance,no_error,4.0,model3,18,37.0
496,2015-01-21 20:00:00,1,170.692019,465.434954,93.524286,36.413510,0,no_maintenance,no_error,4.0,model3,18,38.0
497,2015-01-21 21:00:00,1,162.435715,455.223633,96.753138,46.716736,0,no_maintenance,no_error,4.0,model3,18,39.0
498,2015-01-21 22:00:00,1,165.493259,335.843537,100.454325,35.527658,0,no_maintenance,no_error,4.0,model3,18,40.0


In [9]:
features = (
    "pressure",
    "rotate",
    "volt",
    "vibration"
)
for col in features:
    df9[f"{col}_6h_mean"]=(df9.groupby("machineID")[col].transform(lambda x:x.shift(1).rolling(6,min_periods=3).mean().round(2))).fillna(0)
    df9[f"{col}_6h_std"]=(df9.groupby("machineID")[col].transform(lambda x:x.shift(1).rolling(6,min_periods=3).std().round(2))).fillna(0)
df9.head(20)    

,datetime,machineID,volt,rotate,pressure,vibration,failure,comp,errorID,errors_till_now,...,age,Total_hours_from_last_maintenance,pressure_6h_mean,pressure_6h_std,rotate_6h_mean,rotate_6h_std,volt_6h_mean,volt_6h_std,vibration_6h_mean,vibration_6h_std
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,0,no_maintenance,no_error,0.0,...,18,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,0,no_maintenance,no_error,0.0,...,18,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,0,no_maintenance,no_error,0.0,...,18,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,0,no_maintenance,no_error,0.0,...,18,0.0,94.59,18.93,449.53,67.85,170.03,6.72,40.89,5.87
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,0,no_maintenance,no_error,0.0,...,18,0.0,98.26,17.11,423.69,75.77,168.14,6.67,40.95,4.80
5,2015-01-01 11:00:00,1,172.504839,430.323362,95.927042,35.655017,0,no_maintenance,no_error,0.0,...,18,0.0,100.98,16.02,426.03,65.83,166.03,7.45,37.96,7.88
6,2015-01-01 12:00:00,1,156.556031,499.071623,111.755684,42.753920,0,no_maintenance,no_error,0.0,...,18,0.0,100.14,14.48,426.74,58.90,167.11,7.17,37.57,7.11
7,2015-01-01 13:00:00,1,172.522781,409.624717,101.001083,35.482009,0,no_maintenance,no_error,0.0,...,18,0.0,99.92,14.25,440.17,65.47,163.83,6.65,37.19,6.66
8,2015-01-01 14:00:00,1,175.324524,398.648781,110.624361,45.482287,0,no_maintenance,no_error,0.0,...,18,0.0,100.84,14.08,441.32,64.74,165.44,7.48,35.86,5.93
9,2015-01-01 15:00:00,1,169.218423,460.850670,104.848230,39.901735,0,no_maintenance,no_error,0.0,...,18,0.0,106.74,6.68,419.87,50.22,166.16,8.29,37.75,6.99


In [10]:
df9["future_failure"]=df9.groupby("machineID")["failure"].shift(-6).fillna(0)
df9.tail()

,datetime,machineID,volt,rotate,pressure,vibration,failure,comp,errorID,errors_till_now,...,Total_hours_from_last_maintenance,pressure_6h_mean,pressure_6h_std,rotate_6h_mean,rotate_6h_std,volt_6h_mean,volt_6h_std,vibration_6h_mean,vibration_6h_std,future_failure
877204,2016-01-01 02:00:00,100,179.438162,395.222827,102.290715,50.771941,0,no_maintenance,no_error,31.0,...,188.0,101.42,12.49,474.71,62.56,161.54,7.35,39.91,7.11,0.0
877205,2016-01-01 03:00:00,100,189.617555,446.207972,98.180607,35.123072,0,no_maintenance,no_error,31.0,...,189.0,105.20,6.60,471.35,66.78,164.32,10.42,43.80,5.00,0.0
877206,2016-01-01 04:00:00,100,192.483414,447.816524,94.132837,48.314561,0,no_maintenance,no_error,31.0,...,190.0,105.04,6.78,453.72,53.96,170.35,13.00,43.37,5.68,0.0
877207,2016-01-01 05:00:00,100,165.475310,413.771670,104.081073,44.835259,0,no_maintenance,no_error,31.0,...,191.0,102.80,7.90,437.33,29.80,174.20,15.78,44.97,5.45,0.0
877208,2016-01-01 06:00:00,100,171.336037,496.096870,79.095538,37.845245,0,no_maintenance,no_error,31.0,...,192.0,100.57,3.75,437.29,29.84,176.42,12.71,45.12,5.43,0.0


In [11]:
categorical_cols = ["comp", "errorID","model"]

df9 = pd.get_dummies(
    df9,
    columns=categorical_cols,
    dtype=int
)

In [12]:
df9["datetime"]=pd.to_datetime(df["datetime"])
df9=df9.sort_values(["machineID", "datetime"])

train_parts = []
test_parts=[]

for machine_id,group in df9.groupby("machineID",sort=False):
    n=len(group)
    split_intx =int(n*0.8)
    train_parts.append(group.iloc[:split_intx-6])
    test_parts.append(group.iloc[split_intx:])
train_df = pd.concat(train_parts)
test_df = pd.concat(test_parts)


In [13]:
target = "future_failure"

exclude = [
    "datetime","machineID", "failure", target
]
train_df = train_df.dropna(subset=["future_failure"])
test_df = test_df.dropna(subset=["future_failure"])
X_train = train_df.drop(columns=exclude, axis=1)
y_train = train_df[target]

X_test = test_df.drop(columns=exclude, axis=1)
y_test = test_df[target]



In [14]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [26]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train,y_train)

RandomForestClassifier(class_weight='balanced', min_samples_leaf=2,
                       n_estimators=300, n_jobs=-1, random_state=42)

In [29]:
from sklearn.metrics import classification_report, confusion_matrix

y_prob = model.predict_proba(X_test)[:, 1]

threshold = 0.085
y_pred_new = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred_new))
print(confusion_matrix(y_test, y_pred_new))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    175276
         1.0       0.38      0.89      0.53       210

    accuracy                           1.00    175486
   macro avg       0.69      0.94      0.76    175486
weighted avg       1.00      1.00      1.00    175486

[[174966    310]
 [    23    187]]


In [ ]:
import joblib

joblib.dump(model, "random_forest_model.pkl")